In [1]:
#!/usr/bin/env python3
"""
Visualize per-neuron animate/inanimate contrast weights.

This script computes:

    raw_contrast_j = mean_animate_j - mean_inanimate_j

on raw stimulus-level population responses, and also computes the normalized
directional contrast:

    x_hat_i = x_i / ||x_i||

    mu0 = mean x_hat_i over inanimate stimuli
    mu1 = mean x_hat_i over animate stimuli

    mu0_hat = mu0 / ||mu0||
    mu1_hat = mu1 / ||mu1||

    w = mu1_hat - mu0_hat

Each w_j is the per-neuron contribution to the normalized animate/inanimate
cosine-template classifier.

Outputs
-------
/home/maria/Science/thesis/experiments/007--PoorMansClassifier/
    contrast_distribution_all_neurons/
        contrast_weights.csv
        contrast_summary.json
        hist_normalized_contrast.png
        hist_signed_log_abs_contrast.png
        top_positive_contrast_neurons.png
        top_negative_contrast_neurons.png
        cumulative_abs_contrast.png
        raw_vs_normalized_contrast.png
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# Config
# =============================================================================

BASE_DIR = Path("/home/maria/Science/thesis/experiments/007--PoorMansClassifier")
DATA_DIR = Path("/home/maria/Science/data")

OUT_DIR = BASE_DIR / "contrast_distribution_all_neurons"
OUT_DIR.mkdir(exist_ok=True, parents=True)

NEURAL_FILE = DATA_DIR / "hybrid_neural_responses_reduced.npy"
VIT_FILE = DATA_DIR / "google_vit-base-patch16-224_embeddings_logits.pkl"
VIT_KEY = "natural_scenes"

N_STIMULI = 118
ANIMATE_TOP1_THRESHOLD = 397

PRESENTATION_ORDER = "block"
STIMULUS_IDS_FILE = DATA_DIR / "stimulus_ids.npy"

EPS = 1e-8
TOP_N = 50

# Set True if you want neuron-wise z-scoring before row-normalization.
# I recommend starting with False.
STANDARDIZE_NEURONS = False


# =============================================================================
# Loading
# =============================================================================

def load_neural_presentations() -> np.ndarray:
    if not NEURAL_FILE.exists():
        raise FileNotFoundError(f"Missing neural file: {NEURAL_FILE}")

    X_raw = np.asarray(np.load(NEURAL_FILE, allow_pickle=True))
    print(f"[INFO] Raw neural shape: {X_raw.shape}")

    if X_raw.ndim != 2:
        raise ValueError(f"Expected 2D neural matrix, got {X_raw.shape}")

    n0, n1 = X_raw.shape

    if n0 > n1 and n1 % N_STIMULI == 0:
        print("[INFO] Interpreting raw neural matrix as neurons x presentations.")
        X_pres = X_raw.T
    elif n1 > n0 and n0 % N_STIMULI == 0:
        print("[INFO] Interpreting raw neural matrix as presentations x neurons.")
        X_pres = X_raw
    else:
        raise ValueError(
            f"Could not infer orientation from neural shape {X_raw.shape}. "
            "Expected something like (39209, 118) or (118, 39209)."
        )

    X_pres = X_pres.astype(np.float32, copy=False)
    print(f"[INFO] Presentation-level neural shape: {X_pres.shape}")

    return X_pres


def load_vit_natural_scenes_logits() -> np.ndarray:
    if not VIT_FILE.exists():
        raise FileNotFoundError(f"Missing ViT file: {VIT_FILE}")

    obj = np.load(VIT_FILE, allow_pickle=True)

    if hasattr(obj, "keys"):
        if VIT_KEY not in obj.keys():
            raise KeyError(f"Key {VIT_KEY!r} not found in {VIT_FILE}")
        logits = np.asarray(obj[VIT_KEY])
    elif isinstance(obj, np.ndarray) and obj.dtype == object:
        item = obj.item()
        if VIT_KEY not in item:
            raise KeyError(f"Key {VIT_KEY!r} not found in object dict")
        logits = np.asarray(item[VIT_KEY])
    else:
        raise TypeError(f"Unsupported ViT object type: {type(obj)}")

    if logits.ndim != 2:
        raise ValueError(f"Expected 2D ViT logits, got {logits.shape}")

    if logits.shape[0] != N_STIMULI:
        raise ValueError(f"Expected {N_STIMULI} rows, got {logits.shape[0]}")

    print(f"[INFO] ViT logits shape: {logits.shape}")
    return logits.astype(np.float32, copy=False)


def make_labels_from_vit_logits(logits: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    top1 = np.argmax(logits, axis=1)
    y = (top1 <= ANIMATE_TOP1_THRESHOLD).astype(int)

    print("[INFO] Derived animate/inanimate labels from ViT top-1.")
    print(f"[INFO] Inanimate count: {int((y == 0).sum())}")
    print(f"[INFO] Animate count:   {int((y == 1).sum())}")

    return y, top1


# =============================================================================
# Stimulus averaging
# =============================================================================

def make_presentation_stimulus_ids(n_presentations: int) -> np.ndarray:
    if STIMULUS_IDS_FILE.exists():
        stim_ids = np.load(STIMULUS_IDS_FILE, allow_pickle=True).astype(int).ravel()

        if len(stim_ids) != n_presentations:
            raise ValueError(
                f"{STIMULUS_IDS_FILE} has length {len(stim_ids)}, "
                f"but neural data has {n_presentations} presentations."
            )

        if stim_ids.min() < 0 or stim_ids.max() >= N_STIMULI:
            raise ValueError(
                f"Stimulus IDs must be in [0, {N_STIMULI - 1}], "
                f"got min={stim_ids.min()}, max={stim_ids.max()}."
            )

        print(f"[INFO] Loaded explicit stimulus IDs from {STIMULUS_IDS_FILE}")
        return stim_ids

    if n_presentations % N_STIMULI != 0:
        raise ValueError(
            f"n_presentations={n_presentations} is not divisible by {N_STIMULI}."
        )

    repeats = n_presentations // N_STIMULI

    if PRESENTATION_ORDER == "block":
        stim_ids = np.repeat(np.arange(N_STIMULI), repeats)
    elif PRESENTATION_ORDER == "cycle":
        stim_ids = np.tile(np.arange(N_STIMULI), repeats)
    else:
        raise ValueError("PRESENTATION_ORDER must be either 'block' or 'cycle'.")

    print(
        f"[WARN] No explicit {STIMULUS_IDS_FILE.name} found. "
        f"Assuming PRESENTATION_ORDER={PRESENTATION_ORDER!r}. "
        f"Repeats per stimulus={repeats}."
    )

    return stim_ids.astype(int)


def average_presentations_by_stimulus(X_pres: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    n_presentations, n_neurons = X_pres.shape
    stim_ids = make_presentation_stimulus_ids(n_presentations)

    X_avg = np.zeros((N_STIMULI, n_neurons), dtype=np.float32)
    counts = np.zeros(N_STIMULI, dtype=int)

    for stim_id in range(N_STIMULI):
        mask = stim_ids == stim_id
        counts[stim_id] = int(mask.sum())

        if counts[stim_id] == 0:
            raise ValueError(f"Stimulus {stim_id} has zero presentations.")

        X_avg[stim_id] = X_pres[mask].mean(axis=0)

    print("[INFO] Averaged neural responses by stimulus.")
    print(f"[INFO] Stimulus-averaged neural shape: {X_avg.shape}")
    print(f"[INFO] Presentations per stimulus: min={counts.min()}, max={counts.max()}")

    return X_avg, counts


# =============================================================================
# Cleaning and normalization
# =============================================================================

def clean_features_all_data(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    finite = np.isfinite(X).all(axis=0)
    nonzero_var = np.nanvar(X, axis=0) > 0

    keep = finite & nonzero_var
    kept_original_indices = np.where(keep)[0]

    removed = X.shape[1] - int(keep.sum())
    if removed:
        print(f"[WARN] Removing {removed} non-finite or zero-variance neurons.")

    X_clean = X[:, keep].astype(np.float32, copy=False)

    print(f"[INFO] Clean stimulus-level neural shape: {X_clean.shape}")

    return X_clean, kept_original_indices


def standardize_neurons(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std_safe = np.where(std > EPS, std, 1.0)

    X_z = ((X - mean) / std_safe).astype(np.float32, copy=False)

    return X_z, mean.astype(np.float32), std_safe.astype(np.float32)


def l2_normalize_rows(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms_safe = np.maximum(norms, EPS)
    X_hat = X / norms_safe
    return X_hat.astype(np.float32, copy=False), norms.ravel().astype(np.float32)


def l2_normalize_vector(v: np.ndarray) -> tuple[np.ndarray, float]:
    norm = float(np.linalg.norm(v))
    norm_safe = max(norm, EPS)
    return (v / norm_safe).astype(np.float32, copy=False), norm


# =============================================================================
# Contrast computation
# =============================================================================

def compute_contrasts(
    X_clean: np.ndarray,
    y: np.ndarray,
) -> dict[str, np.ndarray | float]:
    """
    Compute raw and normalized per-neuron animate/inanimate contrast.
    """
    # Raw class means.
    raw_mu0 = X_clean[y == 0].mean(axis=0)
    raw_mu1 = X_clean[y == 1].mean(axis=0)
    raw_contrast = raw_mu1 - raw_mu0

    # Row-normalized class means.
    X_hat, row_norms = l2_normalize_rows(X_clean)

    mu0 = X_hat[y == 0].mean(axis=0)
    mu1 = X_hat[y == 1].mean(axis=0)

    mu0_hat, mu0_norm = l2_normalize_vector(mu0)
    mu1_hat, mu1_norm = l2_normalize_vector(mu1)

    normalized_contrast = mu1_hat - mu0_hat

    return {
        "raw_mu0": raw_mu0,
        "raw_mu1": raw_mu1,
        "raw_contrast": raw_contrast,
        "mu0": mu0,
        "mu1": mu1,
        "mu0_hat": mu0_hat,
        "mu1_hat": mu1_hat,
        "normalized_contrast": normalized_contrast,
        "row_norms": row_norms,
        "template_cosine_similarity": float(np.dot(mu0_hat, mu1_hat)),
        "normalized_contrast_norm": float(np.linalg.norm(normalized_contrast)),
        "raw_contrast_norm": float(np.linalg.norm(raw_contrast)),
        "mu0_norm_before_template_normalization": float(mu0_norm),
        "mu1_norm_before_template_normalization": float(mu1_norm),
    }


def signed_log_abs(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """
    Signed log10 magnitude:

        sign(x) * log10(1 + |x| / eps_scale)

    Uses median nonzero magnitude as scale, not a fixed tiny epsilon,
    so the plot is readable.
    """
    abs_x = np.abs(x)
    nonzero = abs_x[abs_x > 0]

    if len(nonzero) == 0:
        return np.zeros_like(x)

    scale = np.median(nonzero)
    return np.sign(x) * np.log10(1.0 + abs_x / max(scale, eps))


# =============================================================================
# Plotting
# =============================================================================

def save_hist_normalized_contrast(w: np.ndarray) -> None:
    plt.figure(figsize=(8, 5))
    plt.hist(w, bins=100)
    plt.axvline(0, linestyle="--", linewidth=1)
    plt.title("Distribution of normalized animate/inanimate contrast across neurons")
    plt.xlabel("Normalized contrast weight w_j")
    plt.ylabel("Neuron count")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "hist_normalized_contrast.png", dpi=200)
    plt.close()


def save_hist_signed_log_abs_contrast(w: np.ndarray) -> None:
    z = signed_log_abs(w)

    plt.figure(figsize=(8, 5))
    plt.hist(z, bins=100)
    plt.axvline(0, linestyle="--", linewidth=1)
    plt.title("Signed log-magnitude of normalized contrast weights")
    plt.xlabel("sign(w_j) × log-scaled |w_j|")
    plt.ylabel("Neuron count")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "hist_signed_log_abs_contrast.png", dpi=200)
    plt.close()


def save_top_barplots(df: pd.DataFrame, top_n: int = TOP_N) -> None:
    top_pos = df.sort_values("normalized_contrast", ascending=False).head(top_n)
    top_neg = df.sort_values("normalized_contrast", ascending=True).head(top_n)

    plt.figure(figsize=(10, 8))
    plt.barh(
        y=np.arange(len(top_pos)),
        width=top_pos["normalized_contrast"].to_numpy()[::-1],
    )
    plt.yticks(
        np.arange(len(top_pos)),
        top_pos["original_neuron_index"].astype(str).to_numpy()[::-1],
        fontsize=7,
    )
    plt.title(f"Top {top_n} animate-direction neurons")
    plt.xlabel("Normalized contrast weight w_j")
    plt.ylabel("Original neuron index")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "top_positive_contrast_neurons.png", dpi=200)
    plt.close()

    plt.figure(figsize=(10, 8))
    plt.barh(
        y=np.arange(len(top_neg)),
        width=top_neg["normalized_contrast"].to_numpy()[::-1],
    )
    plt.yticks(
        np.arange(len(top_neg)),
        top_neg["original_neuron_index"].astype(str).to_numpy()[::-1],
        fontsize=7,
    )
    plt.title(f"Top {top_n} inanimate-direction neurons")
    plt.xlabel("Normalized contrast weight w_j")
    plt.ylabel("Original neuron index")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "top_negative_contrast_neurons.png", dpi=200)
    plt.close()


def save_cumulative_abs_contrast(w: np.ndarray) -> None:
    abs_sorted = np.sort(np.abs(w))[::-1]
    cumulative = np.cumsum(abs_sorted)
    cumulative_fraction = cumulative / cumulative[-1]

    x = np.arange(1, len(w) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(x, cumulative_fraction)
    plt.xscale("log")
    plt.ylim(0, 1.01)
    plt.title("Cumulative absolute contrast mass")
    plt.xlabel("Top-k neurons by |w_j|, log scale")
    plt.ylabel("Fraction of total |contrast|")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "cumulative_abs_contrast.png", dpi=200)
    plt.close()


def save_raw_vs_normalized_scatter(raw: np.ndarray, normalized: np.ndarray) -> None:
    plt.figure(figsize=(6, 6))
    plt.scatter(raw, normalized, s=4, alpha=0.25)
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.axvline(0, linestyle="--", linewidth=1)
    plt.title("Raw contrast vs normalized directional contrast")
    plt.xlabel("Raw contrast: mean animate - mean inanimate")
    plt.ylabel("Normalized contrast weight w_j")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "raw_vs_normalized_contrast.png", dpi=200)
    plt.close()


# =============================================================================
# Main
# =============================================================================

def main() -> None:
    print("=" * 80)
    print("Loading neural data")
    print("=" * 80)

    X_pres = load_neural_presentations()

    print("=" * 80)
    print("Averaging presentations by stimulus")
    print("=" * 80)

    X_avg, presentation_counts = average_presentations_by_stimulus(X_pres)

    print("=" * 80)
    print("Loading ViT logits and labels")
    print("=" * 80)

    vit_logits = load_vit_natural_scenes_logits()
    y, top1 = make_labels_from_vit_logits(vit_logits)

    if X_avg.shape[0] != len(y):
        raise ValueError(f"X has {X_avg.shape[0]} rows, but y has {len(y)} labels.")

    print("=" * 80)
    print("Cleaning neural features")
    print("=" * 80)

    X_clean, kept_original_neuron_indices = clean_features_all_data(X_avg)

    if STANDARDIZE_NEURONS:
        print("=" * 80)
        print("Standardizing neurons")
        print("=" * 80)
        X_clean, neuron_mean, neuron_std = standardize_neurons(X_clean)
        np.savez_compressed(
            OUT_DIR / "neuron_standardization_stats.npz",
            neuron_mean=neuron_mean,
            neuron_std=neuron_std,
            kept_original_neuron_indices=kept_original_neuron_indices,
        )

    print("=" * 80)
    print("Computing per-neuron contrasts")
    print("=" * 80)

    contrast = compute_contrasts(X_clean, y)

    raw_contrast = contrast["raw_contrast"]
    normalized_contrast = contrast["normalized_contrast"]

    assert isinstance(raw_contrast, np.ndarray)
    assert isinstance(normalized_contrast, np.ndarray)

    abs_w = np.abs(normalized_contrast)

    df = pd.DataFrame(
        {
            "clean_feature_index": np.arange(len(kept_original_neuron_indices)),
            "original_neuron_index": kept_original_neuron_indices,
            "raw_mu_inanimate": contrast["raw_mu0"],
            "raw_mu_animate": contrast["raw_mu1"],
            "raw_contrast": raw_contrast,
            "normalized_contrast": normalized_contrast,
            "abs_normalized_contrast": abs_w,
            "signed_log_abs_normalized_contrast": signed_log_abs(normalized_contrast),
        }
    )

    df = df.sort_values("abs_normalized_contrast", ascending=False)
    df.to_csv(OUT_DIR / "contrast_weights.csv", index=False)

    print("=" * 80)
    print("Saving plots")
    print("=" * 80)

    save_hist_normalized_contrast(normalized_contrast)
    save_hist_signed_log_abs_contrast(normalized_contrast)
    save_top_barplots(df, top_n=TOP_N)
    save_cumulative_abs_contrast(normalized_contrast)
    save_raw_vs_normalized_scatter(raw_contrast, normalized_contrast)

    n_positive = int((normalized_contrast > 0).sum())
    n_negative = int((normalized_contrast < 0).sum())
    n_zero = int((normalized_contrast == 0).sum())

    # How concentrated is the axis?
    sorted_abs = np.sort(abs_w)[::-1]
    cumsum = np.cumsum(sorted_abs)
    total = cumsum[-1]

    def k_for_fraction(frac: float) -> int:
        return int(np.searchsorted(cumsum / total, frac) + 1)

    summary = {
        "experiment": "contrast_distribution_all_neurons",
        "description": (
            "Computes raw mean animate-inanimate contrast and normalized "
            "directional class-template contrast across all cleaned neurons."
        ),
        "neural_file": str(NEURAL_FILE),
        "vit_file": str(VIT_FILE),
        "vit_key": VIT_KEY,
        "n_stimuli": int(N_STIMULI),
        "stimulus_averaged_shape": list(X_avg.shape),
        "clean_shape": list(X_clean.shape),
        "n_clean_neurons": int(X_clean.shape[1]),
        "standardize_neurons": bool(STANDARDIZE_NEURONS),
        "class_counts": {
            "inanimate": int((y == 0).sum()),
            "animate": int((y == 1).sum()),
        },
        "presentation_order_assumption": PRESENTATION_ORDER,
        "used_explicit_stimulus_ids": bool(STIMULUS_IDS_FILE.exists()),
        "min_presentations_per_stimulus": int(presentation_counts.min()),
        "max_presentations_per_stimulus": int(presentation_counts.max()),
        "template_cosine_similarity": float(contrast["template_cosine_similarity"]),
        "normalized_contrast_norm": float(contrast["normalized_contrast_norm"]),
        "raw_contrast_norm": float(contrast["raw_contrast_norm"]),
        "mu0_norm_before_template_normalization": float(
            contrast["mu0_norm_before_template_normalization"]
        ),
        "mu1_norm_before_template_normalization": float(
            contrast["mu1_norm_before_template_normalization"]
        ),
        "normalized_contrast_distribution": {
            "min": float(np.min(normalized_contrast)),
            "q001": float(np.quantile(normalized_contrast, 0.001)),
            "q01": float(np.quantile(normalized_contrast, 0.01)),
            "q05": float(np.quantile(normalized_contrast, 0.05)),
            "median": float(np.median(normalized_contrast)),
            "q95": float(np.quantile(normalized_contrast, 0.95)),
            "q99": float(np.quantile(normalized_contrast, 0.99)),
            "q999": float(np.quantile(normalized_contrast, 0.999)),
            "max": float(np.max(normalized_contrast)),
            "mean": float(np.mean(normalized_contrast)),
            "std": float(np.std(normalized_contrast, ddof=1)),
        },
        "sign_counts": {
            "positive": n_positive,
            "negative": n_negative,
            "zero": n_zero,
        },
        "contrast_concentration": {
            "top_k_for_25pct_abs_mass": k_for_fraction(0.25),
            "top_k_for_50pct_abs_mass": k_for_fraction(0.50),
            "top_k_for_75pct_abs_mass": k_for_fraction(0.75),
            "top_k_for_90pct_abs_mass": k_for_fraction(0.90),
        },
        "output_files": {
            "contrast_weights_csv": str(OUT_DIR / "contrast_weights.csv"),
            "hist_normalized_contrast_png": str(OUT_DIR / "hist_normalized_contrast.png"),
            "hist_signed_log_abs_contrast_png": str(
                OUT_DIR / "hist_signed_log_abs_contrast.png"
            ),
            "top_positive_png": str(OUT_DIR / "top_positive_contrast_neurons.png"),
            "top_negative_png": str(OUT_DIR / "top_negative_contrast_neurons.png"),
            "cumulative_abs_contrast_png": str(OUT_DIR / "cumulative_abs_contrast.png"),
            "raw_vs_normalized_png": str(OUT_DIR / "raw_vs_normalized_contrast.png"),
        },
    }

    with open(OUT_DIR / "contrast_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("=" * 80)
    print("Summary")
    print("=" * 80)
    print(json.dumps(summary, indent=2))

    print("=" * 80)
    print(f"Done. Results saved to: {OUT_DIR}")
    print("=" * 80)


if __name__ == "__main__":
    main()

Loading neural data
[INFO] Raw neural shape: (39209, 118)
[INFO] Interpreting raw neural matrix as neurons x presentations.
[INFO] Presentation-level neural shape: (118, 39209)
Averaging presentations by stimulus
[WARN] No explicit stimulus_ids.npy found. Assuming PRESENTATION_ORDER='block'. Repeats per stimulus=1.
[INFO] Averaged neural responses by stimulus.
[INFO] Stimulus-averaged neural shape: (118, 39209)
[INFO] Presentations per stimulus: min=1, max=1
Loading ViT logits and labels
[INFO] ViT logits shape: (118, 1000)
[INFO] Derived animate/inanimate labels from ViT top-1.
[INFO] Inanimate count: 55
[INFO] Animate count:   63
Cleaning neural features
[INFO] Clean stimulus-level neural shape: (118, 39209)
Computing per-neuron contrasts
Saving plots
Summary
{
  "experiment": "contrast_distribution_all_neurons",
  "description": "Computes raw mean animate-inanimate contrast and normalized directional class-template contrast across all cleaned neurons.",
  "neural_file": "/home/maria